## BUA 340 -- PS3 (IV) Python starter

Use this notebook to answer **Questions 17-20** on the D2L problem set.

### Scenario

A 1,200-employee tech company ran a 12-month **remote-work pilot**. The program was oversubscribed, so HR ran a **lottery** among applicants: roughly half drew a guaranteed remote slot. Most winners took it; a few declined. A handful of non-winners worked remotely anyway through manager exceptions. Twelve months later HR measured **employee engagement** on a 0-100 survey.

| Role | Symbol | Column | Meaning |
|---|---|---|---|
| Outcome | Y | `engagement` | Engagement score after the pilot |
| Treatment | D | `remote` | 1 if the employee actually worked remotely, 0 otherwise |
| Instrument | Z | `remote_offer` | 1 if the employee won the lottery, 0 if not |
| Control | X1 | `tenure_years` | Years at the company |
| Control | X2 | `manager_rating` | Last manager review (1-5 scale) |
| Control | X3 | `team_size` | Size of the employee's team |

The **hidden confounder** is **career ambition** -- it drives both wanting remote work AND underlying engagement. It is *not* in the CSV (a real analyst could not observe it).

### What you need next to this notebook

1. `ps3_iv_data.csv` (ships with this problem set)
2. `iv_toolkit.py` (the same one from the IV lecture -- copy it from the `lec_iv` folder into this folder, **or** edit the import path below)

Then run all cells, read the printed output, and select the matching range on D2L.

### 0. Setup -- imports and load the data

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from iv_toolkit import (
    load_data,
    naive_ols,
    first_stage,
    iv_2sls,
    summary_table,
)

df = load_data('ps3_iv_data.csv')

Loaded 'ps3_iv_data.csv': 1,200 rows x 7 columns
Columns: ['employee_id', 'remote', 'remote_offer', 'tenure_years', 'manager_rating', 'team_size', 'engagement']

First 5 rows:


,employee_id,remote,remote_offer,tenure_years,manager_rating,team_size,engagement
0,0,0,0,3.93,4.69,7,48.08
1,1,1,1,0.03,4.75,11,63.74
2,2,0,0,17.31,2.62,12,53.73
3,3,1,1,7.42,3.26,7,65.56
4,4,0,0,8.16,2.61,3,60.10


In [2]:
# A quick look at the data
df.describe().round(2)

,employee_id,remote,remote_offer,tenure_years,manager_rating,team_size,engagement
count,1200.00,1200.0,1200.00,1200.00,1200.00,1200.00,1200.00
mean,599.50,0.5,0.47,9.06,3.52,8.61,59.20
std,346.55,0.5,0.50,5.08,0.67,3.50,8.60
min,0.00,0.0,0.00,0.03,1.39,3.00,32.74
25%,299.75,0.0,0.00,4.82,3.07,5.00,52.58
50%,599.50,0.5,0.00,9.10,3.53,9.00,59.11
75%,899.25,1.0,1.00,13.24,3.98,12.00,65.39
max,1199.00,1.0,1.00,18.00,5.00,14.00,82.55


### Question 17 -- Naive OLS (no controls)

Run **OLS** (ordinary least squares -- the standard regression of $Y$ on $D$) of `engagement` on `remote` with no controls. Read the coefficient on `remote` in the regression table and pick the range it falls into on D2L.

In [3]:
# -- USER INPUTS --------------------------------------------------------
data       = df
OUTCOME    = 'engagement'    # Y
TREATMENT  = 'remote'        # D
CONTROLS   = []              # start with NO controls
# -----------------------------------------------------------------------

ols_naive = naive_ols(data, y=OUTCOME, d=TREATMENT, controls=CONTROLS)


Naive OLS:  engagement  ~  remote   [HC0 robust SE]
                            OLS Regression Results                            
Dep. Variable:             engagement   R-squared:                       0.535
Model:                            OLS   Adj. R-squared:                  0.534
Method:                 Least Squares   F-statistic:                     1378.
Date:                Thu, 14 May 2026   Prob (F-statistic):          2.05e-201
Time:                        15:40:48   Log-Likelihood:                -3825.4
No. Observations:                1200   AIC:                             7655.
Df Residuals:                    1198   BIC:                             7665.
Df Model:                           1                                         
Covariance Type:                  HC0                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------

### Question 18 -- First-stage F

Run the **first stage** -- the regression of $D$ on $Z$ that tests whether the instrument actually moves the treatment. Here, regress `remote` on the instrument `remote_offer` (no controls). Read the **partial F-statistic on `remote_offer`** -- a single number measuring how strongly $Z$ predicts $D$. The toolkit prints it as a bold colored line at the bottom of the output (not the table's joint F).

In [4]:
# -- USER INPUTS --------------------------------------------------------
INSTRUMENT = 'remote_offer'  # Z
# -----------------------------------------------------------------------

fs = first_stage(data, d=TREATMENT, z=INSTRUMENT, controls=CONTROLS)


First stage:  remote  ~  remote_offer   [HC0 robust SE]
                            OLS Regression Results                            
Dep. Variable:                 remote   R-squared:                       0.261
Model:                            OLS   Adj. R-squared:                  0.260
Method:                 Least Squares   F-statistic:                     425.2
Date:                Thu, 14 May 2026   Prob (F-statistic):           4.35e-81
Time:                        15:40:49   Log-Likelihood:                -689.60
No. Observations:                1200   AIC:                             1383.
Df Residuals:                    1198   BIC:                             1393.
Df Model:                           1                                         
Covariance Type:                  HC0                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------

### Question 19 -- 2SLS (no controls)

Run **2SLS** (two-stage least squares -- the standard way to compute an IV estimate, especially with controls) of `engagement` on `remote`, using `remote_offer` as the instrument, with no controls. Read the coefficient on `remote` from the 2SLS table and pick the range on D2L.

In [5]:
iv_no_ctrl = iv_2sls(data, y=OUTCOME, d=TREATMENT, z=INSTRUMENT, controls=CONTROLS)


2SLS:  engagement  ~  hat(remote)
       (first stage uses remote_offer as the instrument)
                          IV-2SLS Estimation Summary                          
Dep. Variable:             engagement   R-squared:                      0.4135
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4130
No. Observations:                1200   F-statistic:                    79.271
Date:                Thu, May 14 2026   P-value (F-stat)                0.0000
Time:                        15:40:50   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          55.902     0.4340     12

### Question 20 -- 2SLS with controls

Now add the three observable controls -- `tenure_years`, `manager_rating`, `team_size` -- and re-run 2SLS. Compare the coefficient on `remote` to the no-controls estimate from Q19 and pick the option on D2L that best describes what changed.

In [6]:
# -- USER INPUTS --------------------------------------------------------
CONTROLS = ['tenure_years', 'manager_rating', 'team_size']
# -----------------------------------------------------------------------

iv_with_ctrl = iv_2sls(data, y=OUTCOME, d=TREATMENT, z=INSTRUMENT, controls=CONTROLS)


2SLS:  engagement  ~  hat(remote)  +  tenure_years + manager_rating + team_size
       (first stage uses remote_offer as the instrument)
                          IV-2SLS Estimation Summary                          
Dep. Variable:             engagement   R-squared:                      0.4451
Estimator:                    IV-2SLS   Adj. R-squared:                 0.4432
No. Observations:                1200   F-statistic:                    161.29
Date:                Thu, May 14 2026   P-value (F-stat)                0.0000
Time:                        15:40:50   Distribution:                  chi2(4)
Cov. Estimator:                robust                                         
                                                                              
                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------

#### Side-by-side comparison

The true causal effect baked into the data is **+6.0** -- revealed below so you can sanity-check your reading of the regression tables. Look at OLS vs 2SLS to see how much the unobserved confounder (ambition) was distorting OLS, and look at 2SLS-no-ctrl vs 2SLS-with-ctrl to see that controls don't move the IV estimate much (because Z is random).

In [7]:
summary_table({
    'OLS (no controls)':       ols_naive,
    '2SLS (no controls)':      iv_no_ctrl,
    '2SLS (with X controls)':  iv_with_ctrl,
}, true_effect=6.0)


Comparison table:


AttributeError: The '.style' accessor requires jinja2